In [ ]:
import lightkurve as lk
from lightkurve import search_lightcurve
import matplotlib.pyplot as plt
import requests
import pandas as pd
from time import perf_counter
import numpy as np

%matplotlib inline


In [ ]:
def queryMAST(KOI):
    """
    Use the MAST API to get info about the canonical name and KIC of a Kepler Object of Interest (KOI).
    Returns: JSON object with information about object with it's KIC as one of it's fields.
    """
    # Resolve KOI -> Kepler ID / canonical name using Exo.MAST
    info = requests.get(
        "https://exo.mast.stsci.edu/api/v0.1/exoplanets/identifiers/",
        params={"name": KOI},
        timeout=30,
    ).json()
    
    return info


In [ ]:
def fetchStitchedLC(KIC):
    """
    Given a KIC, fetch it's light curve data using LC.
    Returns: Stitched LC object.
    """
    search = lk.search_lightcurve(
        f"KIC {KIC}",
        mission="Kepler",
        author="Kepler",
        cadence="long"
    )

    return search.download_all().stitch().remove_nans()


In [ ]:
df = pd.read_csv(r"../assets/data/kepler-eclipsing-binary-catalog.csv")


In [ ]:
df.head()


In [ ]:
df.drop(columns=["Unnamed: 11"])


In [ ]:
listOfKICs = df['KIC'].astype(int).tolist()


In [ ]:
# print(listOfKICs)
len(listOfKICs)


In [ ]:
sample = listOfKICs[0]
lc = fetchStitchedLC(sample)
lc


In [ ]:
type(lc)


In [ ]:
for i in range(5):
    start = perf_counter()
    KIC = listOfKICs[i]
    lc = fetchStitchedLC(KIC)
    end = perf_counter()
    
    lc.plot()


Test converting the lightcurve into arrays from being an LC object.

In [ ]:
copyDf = df.head(1).copy()


In [ ]:
def getLCArrays(kic):
    # sr = search_lightcurve(f"KIC {kic}", mission="Kepler")
    # lc = sr.download_all().stitch().remove_nans()
    
    lc = fetchStitchedLC(kic)
    
    return pd.Series({
        "time": lc.time.value,
        "flux": lc.flux.value,
        "flux_err": lc.flux_err.value if lc.flux_err is not None else None,
        "n_points": len(lc),
    })

copyDf[["n_points", "time", "flux", "flux_err"]] = copyDf["KIC"].apply(getLCArrays)


In [ ]:
copyDf


In [ ]:
def plotLightCurve(df, kic, with_error=True):
    """
    Plot the light curve for a given KIC from a DataFrame.
    """
    row = df[df["KIC"] == kic]
    
    if row.empty:
        raise ValueError(f"KIC {kic} not found in DataFrame.")
    
    row = row.iloc[0]
    
    t = row["time"]
    f = row["flux"]
    e = row.get("flux_err", None)

    # Plot
    plt.figure(figsize=(20, 5))
    
    if with_error and e is not None:
        plt.errorbar(t, f, yerr=e, fmt='-', linewidth=1)
    else:
        plt.plot(t, f, linewidth=1)
    
    plt.xlabel("Time")
    plt.ylabel(r"Normalized Flux (e$^{-}$ s$^{-1}$)")
    plt.title(f"KIC {kic} Light Curve")
    plt.show()
    

In [ ]:
plotLightCurve(copyDf, 3863594)


In [ ]:
copyDf["n_points"] = copyDf["flux"].apply(len)

# Doesn't really make sense 'cos the LC seems to be normalized already
# df["flux_mean"] = df["flux"].apply(np.mean)
# df["flux_std"] = df["flux"].apply(np.std)


In [ ]:
copyDf
